# 01 — Data import and preprocessing

**Task.** Read the immutable Parquet sources with explicit schemas and construct typed, normalized silver job and resume tables in PySpark.

The notebook is an executable explanation of the production modules. It does not duplicate transformation logic. Raw files are never modified. Resume PII is masked, `Reason_for_decision` is excluded from features, and split assignment groups rows by normalized job-description identity.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'final_project' else Path.cwd().resolve()
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from jobapps.config import load_pipeline_config
from jobapps.pipelines.jobs_pipeline import create_spark_session, run_jobs_pipeline
from jobapps.pipelines.resumes_pipeline import run_resumes_pipeline

CONFIG_PATH = PROJECT_ROOT / 'config' / 'ngram_1pct.yaml'
QUALITY_PATH = PROJECT_ROOT / 'config' / 'data_quality.yaml'
config = load_pipeline_config(CONFIG_PATH)
spark = create_spark_session(config)

## Build silver tables

`run_jobs_pipeline` applies the deterministic catalog sample before joining summaries and skills. `run_resumes_pipeline` preserves all resume rows, masks direct identifiers, assigns labels for audit/evaluation, and creates leakage-safe train/validation/test groups. `write_output=False` keeps this demonstration idempotent.

In [ ]:
silver_jobs, job_metrics, _ = run_jobs_pipeline(
    spark, CONFIG_PATH, QUALITY_PATH, write_output=False
)
silver_resumes, resume_metrics, _ = run_resumes_pipeline(
    spark, config, QUALITY_PATH, write_output=False
)
job_metrics, resume_metrics

In [ ]:
silver_jobs.printSchema()
silver_resumes.groupBy('split').count().orderBy('split').show()
silver_jobs.select('job_link', 'job_title', 'has_summary', 'has_skills').show(10, truncate=60)

## Interpretation

The job sample controls computational scale. Resume splits control development leakage and are independent of catalog sampling. Validation queries are used while choosing retrieval settings; the test split remains locked until the final configuration is frozen.

In [ ]:
spark.stop()